# Prompting for strucutre and setting up a retry method. 

- Define a structure data models for LLM response. 
- Build robust retry mechanisms for validation errors 
- Create reusable function for LLM interactions

In [84]:
# Import neccessary packages 
from pydantic import BaseModel, Field, EmailStr, ValidationError
from typing import List, Literal, Optional
from datetime import date
from dotenv import load_dotenv
import json 
from langchain_groq import ChatGroq
import os 
from groq import Groq

In [2]:
%pip install -qU dotenv

Note: you may need to restart the kernel to use updated packages.


In [85]:
# Load environment variables from API access
load_dotenv()

# Initialize the Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## Define some sample input data

In [ ]:
# Define a JSON string representing user input
user_input_json = '''
{
    "name": "Mausham Kumar",
    "email": "mausham.user@example.com",
    "query": "I forgot my password.",
    "order_number": null,
    "purchase_date": null
}
'''

## Define your UserInput data model¶

In [86]:
# Define UserInput model
class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str
    order_id: Optional[int] = Field(
        None,
        description="5-digit order number (cannot start with 0)",
        ge=10000,
        le=99999
    )
    purchase_date: Optional[date] = None

In [87]:
# Create UserInput instance from JSON data
user_input = UserInput.model_validate_json(user_input_json)

## Create a new data model called CustomerQuery¶

In [88]:
# Define the CustomerQuery model that inherits from UserInput
class CustomerQuery(UserInput):
    priority: str = Field(
        ..., description="Priority level: low, medium, high"
    )
    category: Literal[
        'refund_request', 'information_request', 'other'
    ] = Field(..., description="Query category")
    is_complaint: bool = Field(
        ..., description="Whether this is a complaint"
    )
    tags: List[str] = Field(..., description="Relevant keyword tags")

## Construct a prompt with example output¶

In [89]:
# Create a prompt with generic example data to guide LLM.
example_response_structure = f"""{{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}}"""

In [90]:
# Create prompt with user data and expected JSON structure
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{example_response_structure}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.
"""

print(prompt)


Please analyze this user query
 {
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.



## Define a function to call an LLM and try it with your prompt

In [91]:
def call_llm(prompt: str):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",  # or whatever
        temperature=0,
        # etc
    )
    return resp.choices[0].message.content

In [92]:
# Get response from LLM
response_content = call_llm(prompt)
print(response_content)

{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null,
  "priority": "low",
  "category": "password_request",
  "is_complaint": False,
  "tags": ["password", "support"]
}


### Validate the LLM output using your CustomerQuery model

### Note: the following cell will produce a validation error.

In [93]:
# Attempt to parse the response into CustomerQuery model
valid_data = CustomerQuery.model_validate_json(response_content)

ValidationError: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 9 column 19 [type=json_invalid, input_value='{\n  "name": "Joe User",...assword", "support"]\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid

## Define a function for error handling

In [94]:
# Define a function to validate an LLM response
def validate_with_model(data_model, llm_response):
    try:
        validated_data = data_model.model_validate_json(llm_response)
        print("data validation successful!")
        print(validated_data.model_dump_json(indent=2))
        return validated_data, None
    except ValidationError as e:
        print(f"error validating data: {e}")
        error_message = (
            f"This response generated a validation error: {e}."
        )
        return None, error_message

In [95]:
# Test your validation function with the LLM response
validated_data, validation_error = validate_with_model(
    CustomerQuery, response_content
)

error validating data: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 9 column 19 [type=json_invalid, input_value='{\n  "name": "Joe User",...assword", "support"]\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid


"This response generated a validation error: 1 validation error for CustomerQuery\ncategory\n  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='password_request', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.11/v/literal_error."

## Define a function to create a retry prompt including error details

In [96]:
# Define a function to create a retry prompt with error feedback
def create_retry_prompt(
    original_prompt, original_response, error_message
):
    retry_prompt = f"""
This is a request to fix an error in the structure of an llm_response.
Here is the original request:
<original_prompt>
{original_prompt}
</original_prompt>

Here is the original llm_response:
<llm_response>
{original_response}
</llm_response>

This response generated an error: 
<error_message>
{error_message}
</error_message>

Compare the error message and the llm_response and identify what 
needs to be fixed or removed
in the llm_response to resolve this error. 

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON string.
"""
    return retry_prompt

In [97]:
# Create a retry prompt for validation errors
validation_retry_prompt = create_retry_prompt(
    original_prompt=prompt,
    original_response=response_content,
    error_message=validation_error
)

print(validation_retry_prompt)


This is a request to fix an error in the structure of an llm_response.
Here is the original request:
<original_prompt>

Please analyze this user query
 {
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.

</original_prompt>

Here is the original llm_response:
<llm_response>
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query

## Call the LLM with your retry prompt¶

In [98]:
# Call the LLM with the validation retry prompt
validation_retry_response = call_llm(validation_retry_prompt)
print(validation_retry_response)

{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "I forgot my password.",
    "order_id": null,
    "purchase_date": null,
    "priority": "low",
    "category": "password_request",
    "is_complaint": false,
    "tags": [
        "password",
        "support"
    ]
}


In [99]:
# Attempt to validate retry response from LLM
validated_data, validation_error = validate_with_model(
    CustomerQuery, validation_retry_response
)

error validating data: 1 validation error for CustomerQuery
category
  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='password_request', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/literal_error


In [106]:
validation_error


"This response generated a validation error: 1 validation error for CustomerQuery\ncategory\n  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='password_request', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.11/v/literal_error."

In [103]:
# Create a retry prompt for validation errors
second_validation_retry_prompt = create_retry_prompt(
    original_prompt=validation_retry_prompt,
    original_response=validation_retry_response,
    error_message=validation_error
)

print(validation_retry_prompt)


This is a request to fix an error in the structure of an llm_response.
Here is the original request:
<original_prompt>

Please analyze this user query
 {
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.

</original_prompt>

Here is the original llm_response:
<llm_response>
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query

In [104]:
# Call the LLM with the validation retry prompt
second_validation_retry_response = call_llm(second_validation_retry_prompt)
print(validation_retry_response)

{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "I forgot my password.",
    "order_id": null,
    "purchase_date": null,
    "priority": "low",
    "category": "password_request",
    "is_complaint": false,
    "tags": [
        "password",
        "support"
    ]
}


In [109]:

# Define a function to automatically retry an LLM call multiple times 
def validate_llm_response(prompt, data_model, max_retries=5):
    
    # Initial LLM call 
    response_content = call_llm(prompt=prompt)
    current_prompt = prompt 
    
    # Try to validate with the model 
    # attempt: 0=initial, 1=first retry, ...
    for i in range(max_retries + 1): 
        validated_data, validation_error = validate_with_model(
            data_model, response_content
        )
        
        if validation_error:
            if i < max_retries:
                print(f"retry {i} of {max_retries} failed, trying again...")
            else:
                print(f"Max try reach last error: {validation_error}")
                return None, (f"Max retries reached. Last error: {validation_error}")
            
            validation_retry_prompt = create_retry_prompt(
                original_prompt=current_prompt, 
                original_response=response_content, 
                error_message=validation_error
            )
            
            response_content = call_llm(prompt=validation_retry_prompt)
            
            current_prompt = validation_retry_prompt
            continue
        return validated_data, None
        
            

In [112]:
# Test your complete solution with the original prompt
validated_data, error = validate_llm_response(
    prompt, CustomerQuery
)

error validating data: 1 validation error for CustomerQuery
category
  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='password_reset', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/literal_error
retry 0 of 5 failed, trying again...
data validation successful!
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null,
  "priority": "low",
  "category": "information_request",
  "is_complaint": false,
  "tags": [
    "password",
    "login"
  ]
}


## Have a look at the JSON schema of your CustomerQuery data model

In [113]:
# Investigate the model_json_schema for CustomerQuery
data_model_schema = json.dumps(
    CustomerQuery.model_json_schema(), indent=2
)
print(data_model_schema)

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "email": {
      "format": "email",
      "title": "Email",
      "type": "string"
    },
    "query": {
      "title": "Query",
      "type": "string"
    },
    "order_id": {
      "anyOf": [
        {
          "maximum": 99999,
          "minimum": 10000,
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "5-digit order number (cannot start with 0)",
      "title": "Order Id"
    },
    "purchase_date": {
      "anyOf": [
        {
          "format": "date",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Purchase Date"
    },
    "priority": {
      "description": "Priority level: low, medium, high",
      "title": "Priority",
      "type": "string"
    },
    "category": {
      "description": "Query category",
      "enum"

In [115]:
# Print the original prompt from above
print(prompt)


Please analyze this user query
 {
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.



In [116]:
# Create new prompt with user input and model_json_schema
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching the following schema:
{data_model_schema}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.
"""

In [122]:
# Run your validate_llm_response function with the new prompt
final_analysis, error = validate_llm_response(
    prompt, CustomerQuery
)

error validating data: 7 validation errors for CustomerQuery
name
  Field required [type=missing, input_value={'properties': {'name': {...ery'], 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
email
  Field required [type=missing, input_value={'properties': {'name': {...ery'], 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
query
  Field required [type=missing, input_value={'properties': {'name': {...ery'], 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
priority
  Field required [type=missing, input_value={'properties': {'name': {...ery'], 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
category
  Field required [type=missing, input_value={'properties': {'name': {...ery'], 'type': 'object'}, input_type=dict]
    For further infor

In [ ]:
# Sample json string representation user input 
user_input_json = '''
{
    "name": "Mausham Kumar",
    "email": "kuchbhi@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null
}
'''

In [28]:
# Define UserInput model 
class UserInput(BaseModel):
    name: str = Field(..., description="Full name of the customer")
    email: EmailStr = Field(..., description="Email address of the customer")
    query: str = Field(..., description="Customer's query or issue")
    order_id: Optional[str] = Field(None, description="Order ID if applicable", ge=10000, le=99999 )
    purchase_date: Optional[date] = Field(None, description="Purchase date if applicable")

In [29]:
# Create UserInput instance from the dictionary 
user_input = UserInput.model_validate_json(user_input)
print("User Input:\n", user_input)

User Input:
 name='Mausham Kumar' email='kuchbhi@gmail.com' query='I forgot my password' order_id=None purchase_date=None


- Next we are going to define a new pydantic class for CustomerQuery

In [7]:
class CustomerQuery(UserInput):
    priority: str = Field(..., description="Priority level : Low, Medium, High")
    
    category: Literal['refund_request', 'information_request', 'other'] = Field(..., 
                    description="Whether this is a complaint")
    
    tags: List[str] = Field(..., description="Relevant keyword tags")

# Next thing to do is Construct the prompt 

In [ ]:
# Create a sample model instance with generic example data to guide LLM. 
# This ensure the example differs from the actual user query. 
example_response_structure = f""" {{
    name = "Example  Mausham Kumar", 
    email = "kuchbhi@gmail.com",
    query = " I ordered a new computer montior and it arrived with a broken screen",
    order_id = "12345",
    purchase_date = "2025-10-01",
    priority = "Medium",
    category = "refund_request",
    is_complaint = true,
    tags = ["monitor", "support", "exchange"]
}}

"""

In [32]:
# Create a prompt wiht user data and expect JSON structure. 
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching this exact structure and data types: 
{example_response_structure}

Respond ONLY with valid JSON. Do not include any explanations or other text or formatting before 
or after the JSON object. 
"""

print("Prompt to LLM:\n", prompt)

Prompt to LLM:
 
Please analyze this user query
 {
  "name": "Mausham Kumar",
  "email": "kuchbhi@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure and data types: 
 {
    name = "Example  Mausham Kumar", 
    email = "kuchbhi@gmail.com",
    query = " I ordered a new computer montior and it arrived with a broken screen",
    order_id = "12345",
    purchase_date = "2025-10-01",
    priority = "Medium",
    category = "refund_request",
    is_complaint = true,
    tags = ["monitor", "support", "exchange"]
}



Respond ONLY with valid JSON. Do not include any explanations or other text or formatting before 
or after the JSON object. 



# Define a function to call LLM 

In [33]:
# def call_llm(prompt: str) -> CustomerQuery:
#     response = client.chat(prompt)
#     print("Raw LLM response:\n", response.content)
    
#     try:
#         # Parse the JSON response from the LLM
#         response_json = json.loads(response.content)
        
#         # Validate and create a CustomerQuery instance from the JSON response
#         customer_query = CustomerQuery.model_validate(response_json)
#         return customer_query
    
#     except json.JSONDecodeError as e:
#         print("Failed to parse JSON:", e)
#         raise
    
#     except ValidationError as ve:
#         print("Validation error:", ve)
#         raise


from langchain.schema import HumanMessage

def call_llm(prompt):
    response = client.invoke([HumanMessage(content=prompt)])
    return response.content




In [35]:
# Get the response from LLM 
response_content = call_llm(prompt)
print( response_content)

```json
{
    "name": "Mausham Kumar",
    "email": "kuchbhi@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null,
    "priority": "Medium",
    "category": "account_support",
    "is_complaint": false,
    "tags": ["password_reset", "account_support", "recovery"]
}
```


In [36]:
# Attempt to parse the response into CustomerQuery model 
valid_data = CustomerQuery.model_validate_json(response_content)

ValidationError: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n    "name": ...t", "recovery"]\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid

In [37]:
# Let's define a function to capture this error in pretty way 
def validate_with_model(data_model, llm_response):
    try: 
        valid_data = data_model.model_validate_json(llm_response)
        print("Data validation successful")
        return valid_data, None
    except ValidationError as ve:
        print(f"Error validating data: {ve}")
        error_message = (
            f"This response generated a validation error: {ve}"
        )
    return None, error_message

In [38]:
validate_data, validate_error = validate_with_model(CustomerQuery, response_content)

Error validating data: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n    "name": ...t", "recovery"]\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid


In [60]:
# Send back to LLM and ask to fix this mistake 
def create_retry_prompt(original_prompt, original_response, error_message):
    retry_prompt = f"""
 This is request to fix an error in the structure of an llm_response. 
 Here is the original request:
 <original_prompt>
 {original_prompt}
 </original_prompt>
 
 Here is the original lllm_response: 
<original_response>   
{original_response}
</original_response>

This response generated an error: 
<error_message>
{error_message}
</error_message>

Compare the error message and the llm_response and identify what need to be fixed or remove. 
in the llm_response to resolve this error. 


Response ONLY with valid JSON. Must not include any explanations or other text or formatting before 
or after the JSON object.

Reponse MUST be same structure as this example:
{example_response_structure}
  
"""
    return retry_prompt

In [61]:
# Create a retry prompt for validation error 
validation_retry_prompt = create_retry_prompt(prompt, response_content, validate_error)

print(validation_retry_prompt)


 This is request to fix an error in the structure of an llm_response. 
 Here is the original request:
 <original_prompt>
 
Please analyze this user query
 {
  "name": "Mausham Kumar",
  "email": "kuchbhi@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure and data types: 
 {
    name = "Example  Mausham Kumar", 
    email = "kuchbhi@gmail.com",
    query = " I ordered a new computer montior and it arrived with a broken screen",
    order_id = "12345",
    purchase_date = "2025-10-01",
    priority = "Medium",
    category = "refund_request",
    is_complaint = true,
    tags = ["monitor", "support", "exchange"]
}



Respond ONLY with valid JSON. Do not include any explanations or other text or formatting before 
or after the JSON object. 

 </original_prompt>

 Here is the original lllm_response: 
<original_response>   
```json
{
    "name": "Mausham Kumar",
    "email": "kuc

In [ ]:
# call the llm with the validation retry prompt 
validation_retry_response = call_llm(validation_retry_prompt)
print( validation_retry_response)

```json
{
    "name": "Mausham Kumar",
    "email": "kuchbhi@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null,
    "priority": "Medium",
    "category": "account_support",
    "is_complaint": false,
    "tags": ["password_reset", "account_support", "recovery"]
}
```


In [67]:
# Define a JSON string representing user input
user_input_json = '''
{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "I forgot my password.",
    "order_number": null,
    "purchase_date": null
}
'''

In [68]:
# Define UserInput model
class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str
    order_id: Optional[int] = Field(
        None,
        description="5-digit order number (cannot start with 0)",
        ge=10000,
        le=99999
    )
    purchase_date: Optional[date] = None

In [69]:
# Create UserInput instance from JSON data
user_input = UserInput.model_validate_json(user_input_json)

In [70]:
# Define the CustomerQuery model that inherits from UserInput
class CustomerQuery(UserInput):
    priority: str = Field(
        ..., description="Priority level: low, medium, high"
    )
    category: Literal[
        'refund_request', 'information_request', 'other'
    ] = Field(..., description="Query category")
    is_complaint: bool = Field(
        ..., description="Whether this is a complaint"
    )
    tags: List[str] = Field(..., description="Relevant keyword tags")

In [71]:
# Create a prompt with generic example data to guide LLM.
example_response_structure = f"""{{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}}"""

In [72]:
# Create prompt with user data and expected JSON structure
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{example_response_structure}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.
"""

print(prompt)


Please analyze this user query
 {
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.



In [ ]:
client = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [76]:

# Define a function to call the LLM
def call_llm(prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def call_llm(prompt: str):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",  # or whatever
        temperature=0,
        # etc
    )
    return resp.choices[0].message.content


In [79]:
# Get response from LLM
response_content = call_llm(prompt)
print(response_content)

{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "Forgot password",
    "order_id": null,
    "purchase_date": null,
    "priority": "low",
    "category": "password_reset",
    "is_complaint": False,
    "tags": []
}


In [81]:
!pip install -qU langchain_community